In [1]:
import os
import json
from pydantic import BaseModel
from typing import List, Dict, Optional


In [ ]:
class Task(BaseModel):
    TaskCode: str
    TaskDescription: str

class Node(BaseModel):
    id: str
    filename: str
    project_title: str
    author: str
    table_of_contents: List[str]
    keywords: List[str]
    tasks: List[Task]

class Edge(BaseModel):
    source: str
    target: str
    relationship: str

class Graph(BaseModel):
    nodes: Dict[str, Node]
    edges: List[Edge]
    task_index: Dict[str, List[str]] = {}

    def __init__(self, **data):
        super().__init__(**data)
        self.build_index()

    def build_index(self):
        for node_id, node in self.nodes.items():
            for task in node.tasks:
                if task.TaskCode not in self.task_index:
                    self.task_index[task.TaskCode] = []
                self.task_index[task.TaskCode].append(node_id)

    def update_task(self, old_task_code: str, new_task_code: Optional[str] = None, new_description: Optional[str] = None):
        if old_task_code in self.task_index:
            for node_id in self.task_index[old_task_code]:
                node = self.nodes[node_id]
                for task in node.tasks:
                    if task.TaskCode == old_task_code:
                        if new_task_code:
                            task.TaskCode = new_task_code
                        if new_description:
                            task.TaskDescription = new_description
            if new_task_code:
                self.task_index[new_task_code] = self.task_index.pop(old_task_code)

    def add_or_update_document(self, document_data: Dict):
        node_id = document_data["DocumentFilename"]
        tasks = [Task(**task) for task in document_data["Tasks"]]
        node = Node(
            id=node_id,
            filename=document_data["DocumentFilename"],
            project_title=document_data["ProjectTitle"],
            author=document_data["Author"],
            table_of_contents=document_data["TableOfContents"],
            keywords=document_data["Keywords"],
            tasks=tasks
        )
        self.nodes[node_id] = node
        self.build_index()


In [18]:
with open("sampleDocs.json") as json_file:
    sampleDocs = json.load(json_file)
sampleDocs

[{'DocumentFilename': 'Planning Model Training Using Computer Vision v0.docx',
  'ProjectTitle': 'Planning Model Training Using Computer Vision',
  'Author': 'Hyunsuk',
  'TableOfContents': ['Planning Model Training Using Computer Vision',
   'Phases of the Project and Training Environment Specifications',
   'Introduction',
   'Phases of the Project',
   'Phases of the Project > Model Selection and Development',
   'Phases of the Project > Model Training and Validation',
   'Phases of the Project > Model Testing and Evaluation',
   'Training Environment and Specifications',
   'Training Environment and Specifications > Hardware Specifications',
   'Training Environment and Specifications > Software Specifications',
   'Training Environment and Specifications > Cloud Services',
   'Training Environment and Specifications > Data Management',
   'Conclusion'],
  'Keywords': ['computer vision',
   'model training',
   'model selection',
   'model development',
   'training environment',
 

In [20]:
# print the first document
for doc in sampleDocs:
    print(doc)
    break

{'DocumentFilename': 'Planning Model Training Using Computer Vision v0.docx', 'ProjectTitle': 'Planning Model Training Using Computer Vision', 'Author': 'Hyunsuk', 'TableOfContents': ['Planning Model Training Using Computer Vision', 'Phases of the Project and Training Environment Specifications', 'Introduction', 'Phases of the Project', 'Phases of the Project > Model Selection and Development', 'Phases of the Project > Model Training and Validation', 'Phases of the Project > Model Testing and Evaluation', 'Training Environment and Specifications', 'Training Environment and Specifications > Hardware Specifications', 'Training Environment and Specifications > Software Specifications', 'Training Environment and Specifications > Cloud Services', 'Training Environment and Specifications > Data Management', 'Conclusion'], 'Keywords': ['computer vision', 'model training', 'model selection', 'model development', 'training environment', 'hardware specifications', 'software specifications', 'clo

In [ ]:
# Sample document data
document_data = {
    "DocumentFilename": "Planning Model Training Using Computer Vision v0.docx",
    "ProjectTitle": "Planning Model Training Using Computer",
    "Author": "Hyunsuk",
    "TableOfContents": [
        "Planning Model Training Using Computer Vision",
        "Phases of the Project and Training Environment Specifications",
        "Introduction",
        "Phases of the Project",
        "Phases of the Project > Model Selection and Development",
        "Phases of the Project > Model Training and Validation",
        "Training Environment and Specifications",
        "Training Environment and Specifications > Hardware Specifications",
        "Training Environment and Specifications > Software Specifications",
        "Conclusion"
    ],
    "Keywords": [
        "computer vision",
        "model training",
        "model selection",
        "model development",
        "training environment",
        "hardware specifications",
        "software specifications",
        "cloud services",
        "data management"
    ],
    "Tasks": [
        {
            "TaskCode": "MT0",
            "TaskDescription": "Data Scientist, utilizing machine learning algorithms to train the model on the labeled data."
        }
    ]
}

# Create the graph
graph = Graph(nodes={}, edges=[])


In [4]:
# Add or update the document in the graph
graph.add_or_update_document(document_data)

print(graph.model_dump_json(indent=4))

{
    "nodes": {
        "Planning Model Training Using Computer Vision v0.docx": {
            "id": "Planning Model Training Using Computer Vision v0.docx",
            "filename": "Planning Model Training Using Computer Vision v0.docx",
            "project_title": "Planning Model Training Using Computer",
            "author": "Hyunsuk",
            "table_of_contents": [
                "Planning Model Training Using Computer Vision",
                "Phases of the Project and Training Environment Specifications",
                "Introduction",
                "Phases of the Project",
                "Phases of the Project > Model Selection and Development",
                "Phases of the Project > Model Training and Validation",
                "Training Environment and Specifications",
                "Training Environment and Specifications > Hardware Specifications",
                "Training Environment and Specifications > Software Specifications",
                "Conclusi